# Regex – Cheatsheet für die Prüfung

**Python · Jupyter Notebook · Deutsch**  
Ziel: Reguläre Ausdrücke schnell lesen, schreiben, testen und typische Prüfungsaufgaben lösen.

> **Merksatz:** Regex beschreibt ein **Suchmuster**. In Python verwendet man meist das Modul `re` und **Raw-Strings** wie `r"\d+"`.

## 1. Kurzreferenz: wichtigste Zeichen

| Ausdruck | Bedeutung | Beispiel | Trifft zu |
|---|---|---|---|
| `.` | Beliebiges Zeichen außer Zeilenumbruch (Standard) | `a.c` | `abc`, `a-c` |
| `\d` / `\D` | Ziffer / keine Ziffer | `\d{2}` | `42` |
| `\w` / `\W` | Wortzeichen / kein Wortzeichen | `\w+` | `Hallo_42` |
| `\s` / `\S` | Whitespace / kein Whitespace | `\s+` | Leerzeichen, Tab |
| `[abc]` | Eines aus `a`, `b`, `c` | `[abc]` | `a` |
| `[a-z]` | Zeichen aus einem Bereich | `[A-Z]` | `M` |
| `[^abc]` | Keines aus `a`, `b`, `c` | `[^0-9]` | `x` |
| `^` / `$` | Anfang / Ende einer Zeile oder Zeichenkette (flagabhängig) | `^Hi$` | `Hi` |
| `\A` / `\Z` | Anfang / absolutes Ende der Zeichenkette | `\AHi\Z` | `Hi` |
| `\b` / `\B` | Wortgrenze / keine Wortgrenze | `\bcat\b` | `cat` in `a cat!` |
| `\.` | Literalpunkt | `\.` | `.` |
| `\|` | Literales Pipe-Zeichen | `\|` | `|` |

**Achtung:** `\w` und `\d` erkennen bei Unicode-Strings auch viele nicht-ASCII-Zeichen. Für genau `0–9` verwende `[0-9]`.

## 2. Quantifizierer

| Ausdruck | Bedeutung | Beispiel |
|---|---|---|
| `*` | 0 oder mehr | `ab*` → `a`, `ab`, `abb` |
| `+` | 1 oder mehr | `ab+` → `ab`, `abb` |
| `?` | 0 oder 1 | `colou?r` → `color`, `colour` |
| `{n}` | Genau n-mal | `\d{4}` |
| `{n,}` | Mindestens n-mal | `\d{2,}` |
| `{n,m}` | Zwischen n und m-mal | `\d{2,4}` |
| `*?`, `+?`, `??`, `{n,m}?` | Lazy (so wenig wie möglich) | `<.*?>` |

**Greedy (Standard):** möglichst viel Text erfassen. **Lazy:** möglichst wenig Text erfassen, sodass das Muster noch passt.

In [ ]:
import re

text = "<b>Hallo</b><i>Welt</i>"
print("Greedy:", re.findall(r"<.*>", text))
print("Lazy:  ", re.findall(r"<.*?>", text))

## 3. Gruppen, Alternativen und Referenzen

| Ausdruck | Bedeutung | Beispiel |
|---|---|---|
| `(abc)` | Capturing group | `(ab)+` |
| `(?:abc)` | Non-capturing group | `(?:ab)+` |
| `a|b` | Alternative (ODER) | `Hund|Katze` |
| `(?P<name>...)` | Benannte Gruppe | `(?P<jahr>\d{4})` |
| `\1` | Backreference auf Gruppe 1 | `\b(\w+)\s+\1\b` |
| `(?P=name)` | Backreference auf benannte Gruppe | `(?P<w>\w+)\s+(?P=w)` |

**Wichtig:** `ab|cd` bedeutet `ab` **oder** `cd`. Wenn eine Alternative Teil eines grösseren Musters ist, gruppieren: `^(?:ab|cd)\d+$`.

In [ ]:
s = "Datum: 12.10.2026"
m = re.search(r"(?P<tag>\d{2})\.(?P<monat>\d{2})\.(?P<jahr>\d{4})", s)
print(m.group(0))           # Gesamter Treffer
print(m.group("jahr"))      # Benannte Gruppe
print(m.groups())           # Alle Gruppen als Tupel
print(m.groupdict())        # Benannte Gruppen als Dictionary

## 4. Lookarounds (prüfen, ohne Zeichen zu konsumieren)

| Ausdruck | Bedeutung | Beispiel |
|---|---|---|
| `(?=X)` | Positiver Lookahead: X muss folgen | `\d+(?= CHF)` |
| `(?!X)` | Negativer Lookahead: X darf nicht folgen | `foo(?!bar)` |
| `(?<=X)` | Positiver Lookbehind: X muss davorstehen | `(?<=CHF )\d+` |
| `(?<!X)` | Negativer Lookbehind: X darf nicht davorstehen | `(?<!-)\b\d+\b` |

**Prüfungsfalle:** Bei Python `re` muss ein Lookbehind ein Muster **fester Länge** haben; `(?<=\w+)` führt zu einem Fehler.

In [ ]:
s = "10 CHF, 20 EUR, 30 CHF"
print(re.findall(r"\d+(?= CHF)", s))
print(re.findall(r"(?<=CHF )\d+", "CHF 10; CHF 20"))

In [2]:
text = "3.5 Zimmer"

# Capturing Group
match = re.search(r"\d+(\.\d+)?", text)
print(match.group(0))  # 3.5
print(match.group(1))  # .5

# Non-Capturing Group
match = re.search(r"\d+(?:\.\d+)?", text)
print(match.group(0))  # 3.5
print(match.groups())  # ()

3.5
.5
3.5
()


## 5. Python-`re`: die wichtigsten Funktionen

| Funktion | Zweck | Rückgabe |
|---|---|---|
| `re.search(muster, text)` | Erster Treffer **irgendwo** | Match oder `None` |
| `re.match(muster, text)` | Treffer **am Anfang** | Match oder `None` |
| `re.fullmatch(muster, text)` | **Gesamter** Text muss passen | Match oder `None` |
| `re.findall(muster, text)` | Alle Treffer sammeln | Liste; bei Gruppen ggf. Tupel |
| `re.finditer(muster, text)` | Treffer mit Positionen durchlaufen | Iterator von Match-Objekten |
| `re.sub(muster, ersatz, text)` | Treffer ersetzen | Neuer String |
| `re.split(muster, text)` | An Treffern aufteilen | Liste |
| `re.compile(muster)` | Regex für wiederholte Nutzung vorbereiten | Pattern-Objekt |
| `re.escape(text)` | Sonderzeichen für Literalsuche maskieren | String |

**Tipp:** Für Eingabevalidierung am besten `re.fullmatch()` nutzen (statt nur `search()` oder `match()`).

In [ ]:
text = "Bestellung A-123 und B-456"
print("search:   ", re.search(r"\d+", text).group())
print("findall:  ", re.findall(r"\d+", text))
print("finditer: ", [(m.group(), m.span()) for m in re.finditer(r"\d+", text)])
print("sub:      ", re.sub(r"\d+", "XXX", text))
print("split:    ", re.split(r"\s+", "eins  zwei\tdrei"))
print("fullmatch:", bool(re.fullmatch(r"[A-Z]-\d{3}", "A-123")))

## 6. Flags und Sonderfälle

| Flag | Wirkung |
|---|---|
| `re.I` / `re.IGNORECASE` | Gross-/Kleinschreibung ignorieren |
| `re.M` / `re.MULTILINE` | `^` und `$` beziehen sich auf jede Zeile |
| `re.S` / `re.DOTALL` | `.` erfasst auch Zeilenumbrüche |
| `re.X` / `re.VERBOSE` | Muster mit Leerraum und Kommentaren formatieren |
| `re.A` / `re.ASCII` | ASCII-Verhalten für `\w`, `\d`, `\s`, Grenzen usw. |

Inline-Flags: `(?i)` für case-insensitive (am Musteranfang) oder `(?i:abc)` für eine Teilgruppe.

**Escaping:** `r"\d+"` als Raw-String; für einen echten Punkt `r"\."`, für einen Backslash `r"\\"`.

In [ ]:
print(re.findall(r"^ERROR.*$", "INFO ok\nERROR kaputt\nERROR timeout", flags=re.M))
print(re.findall(r"a.b", "a\nb", flags=re.S))
print(re.findall(r"python", "Python PYTHON", flags=re.I))

## 7. Häufige Prüfungsbeispiele

| Aufgabe | Muster (Python Raw-String) | Hinweise |
|---|---|---|
| Genau fünf ASCII-Ziffern | `r"[0-9]{5}"` | Mit `fullmatch` prüfen |
| Ganze positive Zahl (inkl. 0) | `r"(?:0\|[1-9][0-9]*)"` | Keine führenden Nullen |
| Einfache Dezimalzahl | `r"-?[0-9]+(?:\.[0-9]+)?"` | Punkt als Dezimaltrennzeichen |
| ISO-Datum (Format) | `r"[0-9]{4}-[0-9]{2}-[0-9]{2}"` | Validiert nicht den Kalendertag |
| Schweizer PLZ (Format) | `r"[1-9][0-9]{3}"` | Nur Struktur, nicht existierende PLZ |
| Hexadezimalwert | `r"#[0-9A-Fa-f]{6}"` | Sechs Hex-Zeichen |
| Identifikator | `r"[A-Za-z_][A-Za-z0-9_]*"` | ASCII-Variante |
| E-Mail (vereinfacht) | `r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"` | Keine vollständige E-Mail-Validierung |
| Doppelte Wörter | `r"\b(\w+)\s+\1\b"` | Mit `re.I` auch unabhängig von Gross-/Kleinschreibung |
| Nur Buchstaben A–Z | `r"[A-Za-z]+"` | Umlaute ausgeschlossen |
| Anzahl Zimmer | `r"\d+(?:[.,]\d+)?(?=\s*Zimmer)"` | |
| Preise | `r"(?<=CHF\s)\d+(?:'\d{3})*(?:[\.,]\d+)?"`| 1'441.50, 1441.-, 1441.50 |

> Bei echten Daten sind Regex-Muster häufig nur **Formatprüfungen**. Daten wie das Datum `2026-02-31` müssen zusätzlich fachlich geprüft werden.

In [ ]:
beispiele = [
    (r"[0-9]{5}", "12345"),
    (r"(?:0|[1-9][0-9]*)", "007"),
    (r"[0-9]{4}-[0-9]{2}-[0-9]{2}", "2026-02-31"),
    (r"#[0-9A-Fa-f]{6}", "#A1b2C3"),
]
for muster, wert in beispiele:
    print(f"{wert:12} -> {bool(re.fullmatch(muster, wert))}")

## 8. Prüfungsfallen und Tipps

1. **`search` ≠ `fullmatch`:** `search(r"\d+", "x123y")` trifft zu; `fullmatch` nicht.
2. **`[abc]` ≠ `(abc)`:** Zeichenklasse wählt **ein** Zeichen, Gruppe eine **Sequenz**.
3. **`^` innerhalb von `[]`:** `[^0-9]` negiert die Klasse; ausserhalb markiert `^` den Anfang.
4. **`.` ≠ `\.`:** Punkt ist ein Wildcard, mit Backslash ein wörtlicher Punkt.
5. **`*` erlaubt null Treffer:** `a*` passt auch zu einem leeren String.
6. **`\b` ist Wortgrenze:** nicht das Zeichen für Backspace im Regex-Raw-String.
7. **`$` kann vor einem abschliessenden `\n` passen:** Für ein striktes Ende `\Z` oder `fullmatch`.
8. **Ungeklammerte Alternativen:** `^ab|cd$` entspricht nicht `^(?:ab|cd)$`.
9. **Capture-Gruppen ändern `findall`:** Mit `(\d+)` ist die Gruppe das Ergebnis; mit zwei Gruppen entstehen Tupel.
10. **Regex ist kein vollständiger Parser:** Für komplexes HTML, JSON und echte Datumsvalidierung passende Bibliotheken verwenden.

### Vorgehen in der Prüfung

**1.** Muss der ganze String passen? → `fullmatch`. **2.** Zeichenvorrat festlegen (`[A-Z]`, `\d`, ...). **3.** Wiederholungen festlegen (`+`, `{n,m}`). **4.** Alternativen gruppieren. **5.** Mit gültigen und ungültigen Beispielen testen.

## 9. Mini-Übungen

Versuche zunächst, die Muster selbst zu schreiben. Anschliessend die Lösungszelle ausführen.

1. Finde alle Zahlen mit genau drei Ziffern, die als eigenständiges Wort auftreten.
2. Validiere Codes vom Typ `AB-1234` (zwei Grossbuchstaben, Bindestrich, vier Ziffern).
3. Ersetze mehrere aufeinanderfolgende Leerzeichen durch genau eines.
4. Extrahiere aus `2026-10-12` Jahr, Monat und Tag als benannte Gruppen.
5. Finde doppelte direkt aufeinanderfolgende Wörter wie `und und`.

In [ ]:
# Lösungen
print("1:", re.findall(r"\b[0-9]{3}\b", "12 123 1234 999"))
print("2:", bool(re.fullmatch(r"[A-Z]{2}-[0-9]{4}", "AB-1234")))
print("3:", re.sub(r" {2,}", " ", "Hallo   Welt    !"))
m = re.fullmatch(r"(?P<jahr>[0-9]{4})-(?P<monat>[0-9]{2})-(?P<tag>[0-9]{2})", "2026-10-12")
print("4:", m.groupdict())
print("5:", re.findall(r"\b(\w+)\s+\1\b", "Das ist ist gut und und klar", flags=re.I))

## 10. Schnelltest für eigene Muster

Ändere `muster` und `testwerte`. Nutze `fullmatch`, wenn **der gesamte Wert** den Regeln entsprechen muss.

In [ ]:
muster = r"[A-Z]{2}-[0-9]{4}"
testwerte = ["AB-1234", "A-1234", "ab-1234", "AB-12345", "XX-0000"]

for wert in testwerte:
    print(f"{'OK' if re.fullmatch(muster, wert) else 'NEIN':4}  {wert}")